## Introduction

As a **data analyst**, your objective is to **evaluate how urban mobility relates to economic productivity in major Latin American cities**.

To do this, you will work with real data from the TomTom Traffic Index and OECD economic indicators.

Throughout the project, you will clean, transform, integrate, and analyze the data to identify patterns between **traffic congestion**, **travel times**, and **economic productivity**.

The final goal is to generate actionable insights that can support urban planning and transportation investment decisions.


## 🧩 Step 1: Load and Explore the Data

Before cleaning or combining the data, it is necessary to **become familiar with the structure of both datasets**. At this stage, you will validate that the files load correctly, review their columns and data types, and identify potential inconsistencies.

### 1.1 Load the Datasets

**🎯 Objective:** Load the mobility and economic datasets into Python for analysis.

**Instructions:**
- Import the required libraries.
- Load the traffic and economic datasets using `pd.read_csv()`.
- Store them in the corresponding DataFrames.
- Display the first rows to verify that the data loaded correctly.


In [ ]:
# importar librerías
import pandas as pd
import numpy as nu
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# cargar archivos
traffic = pd.read_csv('/datasets/tomtom_traffic.csv')
eco = pd.read_csv("/datasets/oecd_city_economy.csv")

In [ ]:
# Display the first 5 rows of traffic
traffic.head()

In [ ]:
# Display the first 5 rows of eco
eco.head()

**Tip:** If you do not use `print()`, the table will display more clearly.


---

## 🧩 Step 2: Explore, Clean, and Prepare the Data

Before combining the datasets, inspect their structure, data types, columns, and missing values. Identify the columns that require cleaning and then standardize the column names.


In [ ]:
# Examinar la estructura de traffic
traffic.info()

In the structure of the `traffic` DataFrame, we can observe that:
- The date/time columns are stored as `object` and need to be converted to a date/time data type.
- `UpdateTimeUTCWeekAgo` is also stored as `object` and will require date/time conversion.
- The relevant numeric and categorical fields should be reviewed before analysis.


In [ ]:
# Examinar la estructura de eco
eco.info()

In the structure of the `eco` DataFrame, we can observe that:
- The `City GDP/capita` and `Unemployment %` columns are stored as `object`, so we need to review whether they should be converted to numeric data types.
- The `City` and `Country` columns should be standardized to ensure consistent matching when the datasets are merged.


### 2.2 Rename Columns

**🎯 Objective:** Standardize column names to prevent errors and make it easier to merge the datasets.

**Instructions:**
- Rename the columns using `snake_case`.
- Use clear and consistent technical names throughout the analysis.


In [ ]:
# Standardize traffic column names
traffic = traffic.rename(columns={'Country': 'country', 
                                  'City':'city',
                                  'UpdateTimeUTC': 'update_time_utc',
                                  'JamsDelay': 'jams_delay',
                                  'TrafficIndexLive': 'traffic_index_live',
                                  'JamsLengthInKms': 'jams_length_in_kms',
                                  'JamsCount': 'jams_count',
                                  'TrafficIndexWeekAgo': 'traffic_index_week_ago',
                                  'UpdateTimeUTCWeekAgo': 'update_time_utc_week_ago',
                                  'TravelTimeLivePer10KmsMins': 'travel_time_live_per_10kms_mins',
                                  'TravelTimeHistoricPer10KmsMins': 'travel_time_historic_per_10kms_mins',
                                  'MinsDelay': 'mins_delay'
                                 })

# Verify changes
traffic.columns

In [ ]:
# Standardize eco column names
eco = eco.rename(columns={'Year': 'year',
                          'City': 'city',
                          'Country': 'country', 
                          'City GDP/capita': 'city_gdp_capita', 
                          'Unemployment %': 'unemployment_pct',
                          'PM2.5 (μg/m³)': 'pm25',
                          'Population (M)': 'population_m'})

# Verify changes
eco.columns

### 2.3 Correct Numeric and Date Formats

**🎯 Objective:** Ensure that date and numeric columns use the correct formats so that analyses, calculations, and comparisons can be performed accurately.

**Instructions:**
- Convert the relevant date columns to datetime format.
- Remove non-numeric symbols from economic variables when necessary.
- Convert the cleaned economic variables to numeric data types.
- Verify the resulting data types after conversion.


<details>
<summary>Click to view the hint</summary>
To remove symbols, you can replace them with an empty string.


In [ ]:
# Convert traffic date columns to datetime using pd.to_datetime()
traffic['update_time_utc'] = pd.to_datetime(traffic['update_time_utc'])
traffic['update_time_utc_week_ago'] = pd.to_datetime(traffic['update_time_utc_week_ago'])

# Verify the change
traffic.info()

In [ ]:
# Remove separators and convert numeric columns in eco
# Clean the 'city_gdp_capita' column 
eco['city_gdp_capita'] = eco['city_gdp_capita'] \
                         .str.replace('.', '') \
                         .str.replace(',', '.') \
                         .astype(float)


# Clean the 'unemployment_pct' column 
eco['unemployment_pct'] = eco['unemployment_pct'] \
                         .str.replace('%', '') \
                         .str.replace('.', '') \
                         .str.replace(',', '.') \
                         .astype(float)


# Clean the 'population_m' column 
eco['population_m'] = eco['population_m'] \
                         .str.replace('.', '') \
                         .str.replace(',', '.') \
                         .astype(float)


# Calculate total population in absolute units (multiply by 1,000,000)
eco['population'] = eco['population_m'] * 1000000

# Verify the change
eco.info()
eco.head(3)

---

## 🧩 Step 3: Extract the Year and Filter the Data

Extracting the year makes it possible to filter the information and work only with the most recent and relevant period.

### 3.1 Extract the Year Column and Filter for 2024

**🎯 Objective:** Identify the year of each record and retain only observations corresponding to 2024.

**Instructions:**
- Extract the year from the appropriate date column.
- Create a `year` column.
- Filter the datasets so the analysis uses 2024 data.


In [ ]:
# Extract the year from update_time_utc dates
traffic['year'] = traffic['update_time_utc'].dt.year

# Verify the change
traffic.head(3)

In [ ]:
# Filtra los registros del año 2024
traffic_2024 = traffic[traffic['year'] == 2024].copy()
eco_2024 = eco[eco['year'] == 2024].copy()

# Review dataframes nuevos
display(traffic_2024.head())
display(eco_2024.head())


---

## 🧩 Step 4: Analyze and Summarize Mobility Data

Because the traffic dataset contains **multiple records per city**, you will calculate annual averages by city to simplify the analysis and obtain a clearer view of urban mobility behavior.

**🎯 Objective:** Aggregate traffic indicators by city and year.

**Instructions:**
- Group the traffic data by city and year.
- Calculate the average values of the relevant mobility indicators.
- Reset the index after aggregation.


<details>
<summary>Click to view the hint</summary>
Use `.agg()` to apply mean functions. At the end, reset the index so that the grouping columns remain regular variables rather than indexes.


In [ ]:
# Calculate average traffic indicators by city, country, and year
traffic_city_year_2024 = traffic.groupby(["city", "country", "year"], as_index=False).agg({
    'jams_delay': 'mean',
    'traffic_index_live': 'mean',
    'jams_length_in_kms': 'mean',
    'jams_count': 'mean',
    'mins_delay': 'mean',
    'travel_time_live_per_10kms_mins': 'mean',
    'travel_time_historic_per_10kms_mins': 'mean'
}).reset_index()

# Display the result
traffic_city_year_2024.head()

### 🧠 **Reflection Point**

Excellent work so far!

Now that you have annual averages by city, take a moment to **review them carefully**.

Think about:
- Which city appears to have the highest average traffic time?
- Is it one of the largest cities?
- What might explain the observed mobility pattern?


In [ ]:
traffic_city_year_2024.sort_values(["jams_delay"], ascending=False)

The city with the highest average traffic time is Mexico City.


---

## 🧩 Step 5: Merge Mobility and Economic Data

Combining datasets allows you to analyze how economic indicators relate to urban mobility indicators.

### 5.1 Merge Traffic Data (Main Table) with Economic Indicators

**🎯 Objective:** Combine mobility and economic information into a single analytical dataset.

**Instructions:**
- Merge the traffic and economic DataFrames using the standardized city and year fields.
- Use an **INNER JOIN** so that only cities and years available in both datasets are retained.
- Review the resulting DataFrame to confirm that the merge was successful.


<details>
<summary>Click to view the hint</summary>
Apply an `inner` join to retain only the cities and years present in both datasets.


In [ ]:
# Select key traffic and economic columns
left_cols = ['city','country','year','jams_delay','traffic_index_live',
             'jams_length_in_kms','jams_count','mins_delay',
             'travel_time_live_per_10kms_mins','travel_time_historic_per_10kms_mins']

right_cols = ['city','year','city_gdp_capita','unemployment_pct','pm25','population']

# Use .copy() to create the two reduced datasets
traffic_2024_small = traffic_city_year_2024[left_cols].copy()
eco_2024_small = eco_2024[right_cols].copy()

# Merge datasets
merged = pd.merge(eco_2024_small,traffic_2024_small, on=['city', 'year'], how='left')

# Display the first 5 rows
merged.head()

---

## 🧩 Step 6: Visualization and Relationship Analysis

Now that you have a clean and unified dataset, it is time to **visualize patterns**. The charts will help you understand how economic variables relate to urban mobility indicators.

**🎯 Objective:** Explore distributions, outliers, and potential relationships between congestion and economic productivity.

**Instructions:**
- Create appropriate visualizations for the main mobility and economic variables.
- Use a boxplot to identify potential outliers.
- Use a histogram to review the distribution of GDP per capita.
- Compare mobility and economic indicators across cities.
- Add clear chart titles and axis labels.


**Tip:** Inside the `boxplot()` parentheses, add `showmeans=True` to display the mean on the chart.


In [ ]:
# Create a boxplot to examine the distribution of JamsDelay congestion minutes
# Create the chart

# Calculate the mean to display it in the title
mean_value = merged['jams_delay'].mean()
sns.boxplot(data=merged, x='jams_delay',  showmeans=True)
plt.title(f'JamsDelay Boxplot (2024)\nMean: {mean_value:.2f}')
plt.show()


In [ ]:
# Create a histogram to visualize the distribution of economic productivity (city_gdp_capita)
merged['city_gdp_capita'].hist(bins=5, figsize=(10, 5))
plt.title('City GDP per Capita Histogram (2024)')
plt.show()




In [ ]:
# Bar chart comparing jams_delay and city_gdp_capita by city
merged.plot(kind='bar', y=['jams_delay', 'city_gdp_capita'])
plt.title('Traffic Congestion vs Economic Productivity')
plt.xlabel('Index')
plt.ylabel('Values')
plt.xticks(rotation=90)
plt.show()

**Tip:** Before `plt.show()`, add `plt.xticks(rotation=90)` to rotate the X-axis labels by 90 degrees.


### 🧠 **Reflection**

Excellent work reaching this stage of the analysis. Before moving forward, review your charts and take a moment to consider:

- Do cities with higher GDP per capita also show greater congestion?
- Or does the opposite occur?
- Is there a clear relationship at all?


Write your comments:

- In the boxplot, a very pronounced outlier can be observed, which means that at least one city has a very high congestion level. We could also say that most cities have moderate congestion levels.

- In the histogram, we can observe that most cities are concentrated within an approximate GDP-per-capita range of 8,000 to 16,000. This suggests that most cities have a medium GDP per capita, while a few significantly wealthier cities extend the upper tail of the distribution.

- In the bar chart, `city_gdp_capita` is much larger in scale than `jams_delay`. We can also see that some cities with high GDP have moderate traffic, while others show high congestion. Overall, higher economic productivity does not automatically imply greater traffic congestion.


---

## 🧩 Step 7: Export and Document the Results

In this final stage, you will consolidate your work by saving the clean dataset and creating a summary that documents the project results.

### 7.1 Save the Final Dataset

**🎯 Objective:**
Generate a clean, reproducible CSV containing the relevant columns for future analysis.

**Instructions:**
- Export the `merged` DataFrame using the filename `ladb_mobility_economy_2024_clean.csv`.
- Use `index=False` so the DataFrame index is not included.


In [ ]:
# Exporta el dataset final como CSV
merged.to_csv("ladb_mobility_economy_2024_clean.csv", index=False)

To view or download the generated file:
In the left-side menu, scroll to the bottom and go to the **Export Dataset** section for more information.


---

## ✅ Deliverables

1. **`.ipynb` notebook** containing all cells (code + comments).
2. **Final CSV:** `ladb_mobility_economy_2024_clean.csv`.
3. **Brief executive summary** in Markdown (3–5 paragraphs).


---

# 🧾 Executive Summary (Template)

> Complete this summary after finishing the analysis. Keep it to 3–5 short, clear, and actionable paragraphs.

**Context & Objective:**  
- Answer the central analysis question: What relationship exists between urban mobility (congestion and travel times) and economic productivity (GDP per capita)?
- Briefly explain the key variables used and their relevance to decision-making.

**Data Coverage:**  
- Specify the years analyzed and the number of cities and countries included.

**Methodology (High Level):**  
- Describe the main processes: data cleaning (formats and column standardization).
- Explain the city-year aggregation and the use of an INNER JOIN to integrate traffic and economic data.
- Mention the visual validations used (distributions, outliers, and overall trends).

**Initial Findings:**  
- Summarize the most important patterns between traffic indicators and GDP per capita.
- Highlight anomalies or outliers that may require additional review or deeper analysis.

**Recommendations:**  
Translate the findings into actions: priority cities, the need to validate data sources, additional analysis requirements, or investment proposals.

- Which city—Bogotá, Lima, Buenos Aires, or another city in particular—shows the strongest meaningful relationship between high traffic congestion and low economic productivity indicators, suggesting that it should be prioritized for transportation infrastructure investment?
